# HVG sweep — build the 1k / 2k / 3k variants

Pipeline driver living in `analysis/`, which is why it does not look like a dependency until you read
it: `verify_variants` §9 plots performance against gene-set size and needs these three variants to
exist.

| | |
|---|---|
| **reads** | the same raw sources `1_data` reads |
| **writes** | `<hvg1000\|hvg2000\|hvg3000>/` — the full four-file chain per variant |

**Why it drives `pipeline.*` rather than reimplementing anything.** The sweep's points must be built
exactly as `hvg5000` and `all_genes` are, or the curve's x-axis confounds gene count with code
version. The score and the component count are passed explicitly and checked against the chain's
defaults, so a default drifting raises instead of silently producing a different variant.

⚠️ **Expensive** — three scGPT embeddings. Gated behind `RUN_HVG_SWEEP`, which is the only switch:
`True` means rebuild, including replacing artifacts that already exist.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import inspect

from scripts.layout import DEFAULT_CTRP_SCORE, PipelinePaths
from scripts.preprocessing import add_pca, pipeline

# ⚠️ True for the 13.08.2026 re-embed (Selin). This REVERSES R1, which chose hvg5000 + all_genes
# over "all five" on cost grounds. The reason it is reversed: verify_variants §9 plots performance
# against gene-set size, and with only two variants rebuilt its five points differed in gene count
# AND in code version, so no trend could be read off the curve at all. Rebuilding the three makes
# the sweep like-for-like. **Set this back to False once the sweep exists.**
RUN_HVG_SWEEP = True
SCGPT_PYTHON = '/Users/selin/PycharmProjects/scGPT/.venv/bin/python'
SWEEP_VARIANTS = ['hvg1000', 'hvg2000', 'hvg3000']
PCA_N_COMPS = add_pca.DEFAULT_N_COMPS      # 512, matching the scGPT width

# THIS NOTEBOOK MUST BUILD ITS VARIANTS EXACTLY AS THE NUMBERED CHAIN BUILDS hvg5000 AND all_genes.
# `verify_variants` §9 plots performance against gene-set size across all five, so if the two paths
# construct their variants differently the curve's points differ in construction AS WELL AS in gene
# count -- and no trend can be read off it. That is precisely the defect this rebuild exists to
# remove, so it would be absurd to reintroduce it here.
#
# Until 13.08.2026 the two paths agreed only BY COINCIDENCE: this notebook omitted the score (taking
# `DEFAULT_CTRP_SCORE`, which happened to equal the chain's `SCORE`) and passed `PCA_N_COMPS`, which
# happened to equal `pipeline.pca`'s own default. Change either default and the sweep would silently
# start building variants the chain does not -- with nothing raising. The agreement is now stated and
# asserted instead of inherited.
SCORE = DEFAULT_CTRP_SCORE                 # the chain's 1_data / 3_representations pass this
_pca_default = inspect.signature(pipeline.pca).parameters['n_comps'].default
if PCA_N_COMPS != _pca_default:
    raise ValueError(
        f'PCA_N_COMPS={PCA_N_COMPS} but pipeline.pca defaults to {_pca_default}, which is what the '
        f'numbered chain uses. The sweep would build its variants on a different number of '
        f'components than hvg5000 and all_genes, and verify_variants §9 would compare them as if '
        f'only the gene count differed. Pass the same value in both places, or change both.')

print(f'score={SCORE}  n_comps={PCA_N_COMPS} (matches pipeline.pca default)')
for v in SWEEP_VARIANTS:
    p = PipelinePaths.build(None, v, SCORE)
    print(f'  {v:9s} {"present" if p.targets_h5ad.exists() else "MISSING"}')

score=auc_cc  n_comps=512 (matches pipeline.pca default)
  hvg1000   present
  hvg2000   MISSING
  hvg3000   MISSING


## Build

| | |
|---|---|
| **out** | per variant: raw h5ad → embeddings → targets → splits → PCA |

The same six steps in the same order as the numbered chain, once per variant. `overwrite=True` on the
two guarded steps, because replacing the existing artifacts is the point of running this.

In [2]:
if not RUN_HVG_SWEEP:
    print('RUN_HVG_SWEEP is False -- skipping the heavy scGPT sweep build.')
else:
    for variant in SWEEP_VARIANTS:
        paths = PipelinePaths.build(None, variant, SCORE)
        # ⚠️ The "targets present -> skip" guard is DELIBERATELY GONE (13.08.2026, Selin).
        # It read:
        #     if paths.targets_h5ad.exists():
        #         print(f'{variant}: targets present, skipping.'); continue
        # which made this cell a no-op for exactly the case it is now being run for: all three
        # variants already had targets, built 01:53 on 13.08.2026 by the pre-correction code. With
        # the guard in place, flipping RUN_HVG_SWEEP and passing overwrite=True would still have
        # skipped every variant and reported success -- the same shape of silent no-op this review
        # has been finding all day. RUN_HVG_SWEEP is now the only switch: True means rebuild.
        print('\n' + '=' * 70 + f'\n{variant}\n' + '=' * 70)
        pipeline.fetch(paths)
        # overwrite=True on the two guarded steps, because replacing these artifacts IS the point.
        # convert() attaches total_counts / pct_counts_mt itself, so the rebuilt variants carry the
        # UMI covariates the 01:53 ones lack -- which is half of what made the sweep unreadable.
        pipeline.convert(paths, overwrite=True)
        pipeline.scgpt(paths, SCGPT_PYTHON, overwrite=True)
        pipeline.targets(paths)
        pipeline.splits(paths)
        pipeline.pca(paths, n_comps=PCA_N_COMPS)
        print(f'{variant}: built -> {paths.targets_h5ad}')


hvg1000
[fetch] Zenodo record 21807175
Zenodo record 21807175 -- 'Dataset for drevalpy', DOI 10.5281/zenodo.21807175, published 2026-08-05
  CTRPv2.zip: cached, MD5 verified -- skipping download
  meta.zip: cached, MD5 verified -- skipping download
Cached at /Users/selin/Desktop/OncoTox/data/metadata/drevalpy_CTRPv2_zenodo_21807175
[convert] hvg1000: top-1000 HVGs
Loading expression matrix... (this may take a few minutes and require high RAM)
Loading metadata...
Aligning metadata with expression data...
[qc] reading cached UMI covariates from umi_qc_covariates.csv
[qc] join check: genes detected vs obs['Genes_expressed'] r=1.0000
[qc] added ['total_counts', 'pct_counts_mt'] | median depth 16,733 | median mito 5.61%
Annotating current HGNC symbols...
  hgnc_symbol: 1,129 of 22,722 rows renamed to their current symbol, 23 renames withheld as collisions
Selecting top 1000 highly variable genes...
  Gene count: 22722 -> 1000
Saving to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_S

/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Success! Created AnnData object: AnnData object with n_obs × n_vars = 53513 × 1000
    obs: 'Cell_line', 'Pool_ID', 'Cancer_type', 'Genes_expressed', 'Discrete_cluster_minpts5_eps1.8', 'Discrete_cluster_minpts5_eps1.5', 'Discrete_cluster_minpts5_eps1.2', 'CNA_subclone', 'SkinPig_score', 'EMTI_score', 'EMTII_score', 'EMTIII_score', 'IFNResp_score', 'p53Sen_score', 'EpiSen_score', 'StressResp_score', 'ProtMatu_score', 'ProtDegra_score', 'G1/S_score', 'G2/M_score', 'total_counts', 'pct_counts_mt'
    var: 'hgnc_symbol'
    uns: 'hvg_n_top_genes'
[scgpt] /Users/selin/PycharmProjects/scGPT/.venv/bin/python /Users/selin/PycharmProjects/OncoTox/scripts/preprocessing/gen_embeds.py --input /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg1000/SCP542_CCLE.h5ad --output /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg1000/SCP542_CCLE_scGPT_human_embeddings.h5ad --model-dir /Users/selin/Desktop/OncoTox/scGPT/scGPT_human
/Users/selin/PycharmProjects/scGPT/scgpt/model/mo

/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Done!
[splits] seed=42, regenerate=False
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg1000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...
Found 181 unique cell lines with at least one CTRP drug label across 47440 cells.
  Using frozen split from /Users/selin/PycharmProjects/OncoTox/splits/split_ctrp.csv (181 lines).
Cell Line Split -> train: 126, val: 27, test: 28

Final Cell Split distribution for multi-drug (split_ctrp):
split_ctrp
train         34082
val            6829
test           6529
unassigned     6073
Name: count, dtype: int64

Saving updated AnnData...


...

/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Done!
[splits] seed=42, regenerate=False
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg3000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...
Found 181 unique cell lines with at least one CTRP drug label across 47440 cells.
  Using frozen split from /Users/selin/PycharmProjects/OncoTox/splits/split_ctrp.csv (181 lines).
Cell Line Split -> train: 126, val: 27, test: 28

Final Cell Split distribution for multi-drug (split_ctrp):
split_ctrp
train         34082
val            6829
test           6529
unassigned     6073
Name: count, dtype: int64

Saving updated AnnData...


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Done! Leakage-free grouped splits are permanently saved.
[pca] n_comps=512, seed=42, force=False
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg3000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...
Computing PCA on HVG-filtered counts from /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg3000/SCP542_CCLE.h5ad...
  X_pca_train_ctrp: fitted on 34082 train cells -> shape (53513, 512).
  X_pca (all cells) computed on 3000 genes -> shape (53513, 512).
    52.4% of variance retained.
Saving updated AnnData with X_pca (targets .X unchanged)...
Done! You can now run baseline training.
hvg3000: built -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg3000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)
